# Neural Activity Animation â€” PCA Scatter Across All SelfAttention.o Layers

Animates a 2D PCA scatter plot cycling through all 48 `SelfAttention.o` layers in forward-pass
order. At each frame, **correct** (blue) and **hallucinated** (red) scan groups are projected
into 2D PCA space. The viewer watches the two clouds separate (or not) as the network gets deeper.

Key design choices:
- **Per-layer PCA fit** (not a single global PCA): captures the intrinsic 2D structure at each layer.
- **Global axis limits**: computed across all 48 layers before animation so clouds visually
  move and separate rather than just re-scaling per frame.
- **No cross-validation**: PCA is fit on all 100 samples per layer â€” purely for visualisation.
- **repeat=True**: animation loops continuously (useful for LinkedIn posts).
- **GIF output**: most portable format for embedding; also renderable inline via `to_jshtml()`.

## Cell 1 â€” Imports

In [1]:
import sys, os, re

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from scipy.spatial import ConvexHull
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display

from neuralsignal.backend.ns_backend import NSBackend

print("Imports OK")

E:\Programming\neuralsignal\neuralsignal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## Cell 2 â€” Config

In [2]:
APPLICATION_NAME     = "redis"
SUB_APPLICATION_NAME = "quora_duplicates_t5_large"

GROUP_SIZE   = 50          # scans per class
RANDOM_STATE = 42

GROUND_TRUTH_CORRECT      = "0"
GROUND_TRUTH_HALLUCINATED = "1"

FPS        = 3     # frames per second (slow enough to read each layer)
SAVE_GIF   = True  # set False to skip saving
GIF_PATH   = "neural_activity_animation.gif"

## Cell 3 â€” Connect to backend & load scans

In [3]:
import tempfile
from neuralsignal.core.modules.neuralsignal_config import sdk_config

backend_cfg = sdk_config.get_backend_config()
backend_cfg["scan_cache_directory"] = tempfile.gettempdir()

cfg = {
    "application_name": APPLICATION_NAME,
    "sub_application_name": SUB_APPLICATION_NAME,
    "backend_config": backend_cfg,
}
be = NSBackend(cfg)

def load_group(be, ground_truth_value, n):
    query = {"ground_truth": ground_truth_value}
    cursor = be.query(query).limit(n)
    scans = []
    for doc in cursor:
        try:
            scans.append(be.deserialize_scan(doc))
        except Exception as e:
            print(f"  skipping {doc.get('_id')}: {e}")
    return scans

print(f"Loading correct group (gt={GROUND_TRUTH_CORRECT}) ...")
group_correct = load_group(be, GROUND_TRUTH_CORRECT, GROUP_SIZE)
print(f"  loaded {len(group_correct)} scans")

print(f"Loading hallucinated group (gt={GROUND_TRUTH_HALLUCINATED}) ...")
group_hallucinated = load_group(be, GROUND_TRUTH_HALLUCINATED, GROUP_SIZE)
print(f"  loaded {len(group_hallucinated)} scans")

Loading correct group (gt=0) ...
  loaded 50 scans
Loading hallucinated group (gt=1) ...
  loaded 50 scans


## Cell 4 â€” Discover SelfAttention.o layers

Inspects the first scan to find all `SelfAttention.o` layers in forward-pass order
and builds short display labels from the block index embedded in each layer name.

In [4]:
def get_sa_layers(scan):
    """Return [(layer_id, full_name, short_label), ...] in forward-pass order."""
    lid_to_name = scan["layer_id_to_name"]
    enc_idx = dec_idx = 0
    rows = []
    for lid in scan["layer_order"]:
        name = lid_to_name[lid]
        if "SelfAttention.o" not in name:
            continue
        is_enc = "encoder" in name.lower()
        m = re.search(r'\.(\d+)\.', name)
        if m:
            label = f"{'enc' if is_enc else 'dec'}.{m.group(1)}"
        else:
            if is_enc:
                label = f"enc.{enc_idx}"
                enc_idx += 1
            else:
                label = f"dec.{dec_idx}"
                dec_idx += 1
        rows.append((lid, name, label))
    return rows

sa_layers = get_sa_layers(group_correct[0])
print(f"Found {len(sa_layers)} SelfAttention.o layers")

Found 48 SelfAttention.o layers


## Cell 5 â€” Helper functions

In [5]:
def extract_vector(scan, layer_id):
    tensor = scan["outputs"][layer_id]
    return tensor[-1, :].float()   # last token, shape (hidden_dim,)

def build_matrix(scans, layer_id, expected_dim=None):
    vecs, skipped = [], 0
    for scan in scans:
        try:
            v = extract_vector(scan, layer_id)
            if expected_dim is not None and v.shape[0] != expected_dim:
                skipped += 1
                continue
            vecs.append(v)
        except Exception:
            skipped += 1
    return torch.stack(vecs), skipped

def draw_convex_hull(ax, points, color, alpha=0.12):
    """Draw a filled convex hull around a 2D point cloud."""
    if len(points) < 3:
        return
    try:
        hull = ConvexHull(points)
        vertices = np.append(hull.vertices, hull.vertices[0])   # close the loop
        ax.fill(points[vertices, 0], points[vertices, 1],
                color=color, alpha=alpha, linewidth=0)
        ax.plot(points[vertices, 0], points[vertices, 1],
                color=color, alpha=0.35, linewidth=1)
    except Exception:
        pass

print("Helpers defined")

Helpers defined


## Cell 6 â€” Pre-compute per-layer PCA projections

For each layer:
1. Extract activation matrices for correct and hallucinated groups.
2. Fit `PCA(n_components=2)` on the combined 100-sample matrix.
3. Project both groups into 2D.
4. Accumulate global axis bounds so the animation uses fixed axes across all frames.

In [6]:
frames_data = []   # list of dicts, one per layer

global_xmin, global_xmax = +np.inf, -np.inf
global_ymin, global_ymax = +np.inf, -np.inf

for i, (lid, name, label) in enumerate(sa_layers):
    vecs_c, _  = build_matrix(group_correct,      lid)
    vecs_h, sk = build_matrix(group_hallucinated, lid, expected_dim=vecs_c.shape[1])
    if sk > 0:
        print(f"  [{label}] skipped {sk} hallucinated scans (dim mismatch)")

    X = np.vstack([vecs_c.numpy(), vecs_h.numpy()])   # (N_c + N_h, D)

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    pca.fit(X)

    proj_c = pca.transform(vecs_c.numpy())   # (N_c, 2)
    proj_h = pca.transform(vecs_h.numpy())   # (N_h, 2)

    var = pca.explained_variance_ratio_ * 100

    frames_data.append({
        "label":       label,
        "frame_index": i,
        "proj_correct":      proj_c,
        "proj_hallucinated": proj_h,
        "var1": var[0],
        "var2": var[1],
    })

    # Accumulate global axis bounds
    all_pts = np.vstack([proj_c, proj_h])
    global_xmin = min(global_xmin, all_pts[:, 0].min())
    global_xmax = max(global_xmax, all_pts[:, 0].max())
    global_ymin = min(global_ymin, all_pts[:, 1].min())
    global_ymax = max(global_ymax, all_pts[:, 1].max())

    print(f"  [{i:>2}] {label:<10}  PC1={var[0]:.1f}%  PC2={var[1]:.1f}%")

# Add 5% padding on each side
x_pad = (global_xmax - global_xmin) * 0.05
y_pad = (global_ymax - global_ymin) * 0.05
global_xmin -= x_pad;  global_xmax += x_pad
global_ymin -= y_pad;  global_ymax += y_pad

print(f"\nPre-computed {len(frames_data)} frames")
print(f"Global X range: [{global_xmin:.2f}, {global_xmax:.2f}]")
print(f"Global Y range: [{global_ymin:.2f}, {global_ymax:.2f}]")

  [ 0] enc.0       PC1=22.4%  PC2=13.3%
  [ 1] enc.1       PC1=32.9%  PC2=10.7%
  [ 2] enc.2       PC1=13.4%  PC2=11.6%
  [ 3] enc.3       PC1=14.9%  PC2=9.4%
  [ 4] enc.4       PC1=12.8%  PC2=10.2%
  [ 5] enc.5       PC1=21.6%  PC2=9.0%
  [ 6] enc.6       PC1=10.3%  PC2=7.7%
  [ 7] enc.7       PC1=11.2%  PC2=8.6%
  [ 8] enc.8       PC1=13.0%  PC2=12.7%
  [ 9] enc.9       PC1=11.9%  PC2=9.1%
  [10] enc.10      PC1=12.5%  PC2=8.9%
  [11] enc.11      PC1=9.4%  PC2=7.5%
  [12] enc.12      PC1=17.1%  PC2=7.7%
  [13] enc.13      PC1=12.2%  PC2=11.3%
  [14] enc.14      PC1=16.6%  PC2=12.2%
  [15] enc.15      PC1=15.3%  PC2=10.7%
  [16] enc.16      PC1=16.1%  PC2=7.7%
  [17] enc.17      PC1=34.2%  PC2=9.2%
  [18] enc.18      PC1=35.8%  PC2=10.9%
  [19] enc.19      PC1=19.6%  PC2=14.9%
  [20] enc.20      PC1=31.3%  PC2=8.5%
  [21] enc.21      PC1=46.2%  PC2=8.4%
  [22] enc.22      PC1=66.5%  PC2=7.9%
  [23] enc.23      PC1=52.3%  PC2=17.4%
  [24] dec.0       PC1=nan%  PC2=nan%
  [25] dec.1    

E:\Programming\neuralsignal\neuralsignal\.venv\Lib\site-packages\sklearn\decomposition\_pca.py:779: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = self.explained_variance_ / total_var


  [39] dec.15      PC1=49.6%  PC2=14.4%
  [40] dec.16      PC1=74.5%  PC2=5.7%
  [41] dec.17      PC1=60.6%  PC2=12.6%
  [42] dec.18      PC1=61.4%  PC2=12.4%
  [43] dec.19      PC1=58.6%  PC2=12.7%
  [44] dec.20      PC1=81.5%  PC2=8.7%
  [45] dec.21      PC1=68.6%  PC2=16.8%
  [46] dec.22      PC1=70.7%  PC2=12.9%
  [47] dec.23      PC1=59.3%  PC2=24.9%

Pre-computed 48 frames
Global X range: [-588.70, 875.32]
Global Y range: [-427.79, 406.33]


## Cell 7 â€” Build and display animation

Each frame shows the 2D PCA scatter for one `SelfAttention.o` layer.
Axes are fixed to global bounds so the two clouds appear to move and separate
rather than simply re-scaling each frame.

In [7]:
fig, ax = plt.subplots(figsize=(8, 7))
n_layers = len(sa_layers)

def update(frame_idx):
    ax.clear()
    d = frames_data[frame_idx]
    proj_c = d["proj_correct"]
    proj_h = d["proj_hallucinated"]

    # Scatter points
    ax.scatter(proj_c[:, 0], proj_c[:, 1],
               c="steelblue", s=35, alpha=0.65, linewidths=0,
               label=f"Correct (n={len(proj_c)})")
    ax.scatter(proj_h[:, 0], proj_h[:, 1],
               c="crimson",   s=35, alpha=0.65, linewidths=0,
               label=f"Hallucinated (n={len(proj_h)})")

    # Convex hulls
    draw_convex_hull(ax, proj_c, "steelblue")
    draw_convex_hull(ax, proj_h, "crimson")

    # Centroids
    cf = proj_c.mean(axis=0)
    ch = proj_h.mean(axis=0)
    ax.scatter(*cf, c="steelblue", s=220, marker="*", zorder=5,
               edgecolors="white", linewidths=0.8)
    ax.scatter(*ch, c="crimson",   s=220, marker="*", zorder=5,
               edgecolors="white", linewidths=0.8)

    # Fixed axis limits (global bounds)
    ax.set_xlim(global_xmin, global_xmax)
    ax.set_ylim(global_ymin, global_ymax)

    ax.set_xlabel(f"PC1  ({d['var1']:.1f}% variance)", fontsize=10)
    ax.set_ylabel(f"PC2  ({d['var2']:.1f}% variance)", fontsize=10)
    ax.set_title(
        f"Layer [{frame_idx + 1}/{n_layers}] â€” {d['label']}  "
        f"|  PC1={d['var1']:.1f}%  PC2={d['var2']:.1f}%",
        fontsize=12,
    )
    ax.legend(fontsize=9, loc="upper right", framealpha=0.85)
    ax.grid(linestyle="--", alpha=0.3)

anim = FuncAnimation(
    fig, update,
    frames=n_layers,
    interval=1000 // FPS,
    blit=False,
    repeat=True,
)

# Display inline
display(HTML(anim.to_jshtml()))

# Save as GIF
if SAVE_GIF:
    anim.save(GIF_PATH, writer=PillowWriter(fps=FPS))
    print(f"Saved â†’ {GIF_PATH}")

plt.close(fig)

INFO:matplotlib.animation:Animation.save using <class 'matplotlib.animation.HTMLWriter'>


INFO:matplotlib.animation:Animation.save using <class 'matplotlib.animation.PillowWriter'>


Saved â†’ neural_activity_animation.gif
